# Jupyter tour: magics, plotting, reproducibility

Companion to `notes/06-reproducibility-jupyter.md`. Execute top to bottom with
`jupyter nbconvert --to notebook --execute tour.ipynb`; every cell is deterministic.

The kernel is a separate process (ZeroMQ messages carry code in, results out); the
notebook file is JSON holding source *and* outputs, which is why stale outputs and
out-of-order execution are the classic hazards.

In [ ]:
%matplotlib inline
import sys, platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
print(sys.version.split()[0], platform.machine(), "numpy", np.__version__, "matplotlib", matplotlib.__version__)

## Line and cell magics

`%name` acts on the rest of the line, `%%name` on the whole cell. They are IPython
features, not Python: a plain `python script.py` would fail on them.

In [ ]:
%time s = sum(range(2_000_000))
%timeit -n 3 -r 3 np.sort(np.random.default_rng(0).random(200_000))

In [ ]:
%%time
# cell magic: times the whole cell
x = np.linspace(0, 1, 1_000_001)
y = np.cumsum(np.sin(x)) / x.size
y[-1]

In [ ]:
%who_ls ndarray

In [ ]:
# %lsmagic lists everything; %quickref is the cheat sheet; ?obj shows the docstring, ??obj the source
import inspect
print(inspect.getsource(np.trapezoid).splitlines()[0])   # what ??np.trapezoid would show, without the pager

## Seeds and reproducibility

Use a `Generator` created from an explicit seed and pass it down; never rely on
global state (`np.random.seed`) that another cell can silently change.

In [ ]:
rng = np.random.default_rng(2026)
sample = rng.normal(size=5)
again = np.random.default_rng(2026).normal(size=5)
assert np.array_equal(sample, again)
sample

In [ ]:
# floating-point reproducibility: the summation ORDER changes the last bits
import math
a = np.random.default_rng(1).random(10**6)
pairwise = a.sum()                 # numpy: pairwise summation, error O(log n) eps
sequential = np.cumsum(a)[-1]      # left-to-right, error O(n) eps
exact = math.fsum(a)               # correctly rounded reference
print(f"{pairwise!r}\n{sequential!r}\n{exact!r}")
print("pairwise == sequential:", pairwise == sequential,
      " |err| pairwise, sequential:", abs(pairwise - exact), abs(sequential - exact))

## Plotting inline

With the inline backend a figure is rendered to PNG at the end of the cell and stored
in the notebook JSON (`outputs[].data['image/png']`).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3), constrained_layout=True)
t = np.linspace(0, 4 * np.pi, 400)
for k in (1, 2):
    ax.plot(t, np.exp(-t / 8) * np.sin(k * t), label=f"k={k}")
ax.set(xlabel="t", ylabel="damped sine"); ax.legend()
import tempfile, pathlib
out = pathlib.Path(tempfile.gettempdir()) / "tour_damped.png"
fig.savefig(out, dpi=120)   # persist explicitly; the inline PNG lives only in the .ipynb
print("saved", out)          # the figure itself is displayed automatically at the end of the cell

## Environment record

Print what the notebook ran with, so a reader can rebuild the environment
(`uv pip freeze > requirements.txt` / `uv lock` gives the exact pinned set).

In [ ]:
import importlib.metadata as im
{name: im.version(name) for name in ["numpy", "scipy", "matplotlib", "pandas", "ipykernel"]}

## Hygiene checklist

1. Restart kernel and run all before sharing (`Kernel > Restart & Run All`).
2. Seed every random source explicitly; record versions.
3. Move stable code into a module (`src/py/...`) and import it; keep the notebook for exploration and narrative.
4. Clear outputs or use `nbstripout` before committing, unless the outputs *are* the deliverable.
5. Long computations: cache results to disk (`np.save`) and load them in a separate cell.